# Phase 4: Modelling (Part 1 - Baseline Random Forest All Features)
**Client:** Crédit Nationale Azur
**Objective:** Build an initial baseline Random Forest classifier using ALL available features (no feature selection) to establish a benchmark for comparison.

In this notebook, we load the cleaned data, apply strict data leakage prevention during our train/test split, and encode our categorical variables before evaluating the model's performance on the imbalanced dataset.

In [21]:
# Required Base Imports
import pandas as pd
import numpy as np
import joblib
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, precision_score, recall_score, f1_score
from sklearn.ensemble import RandomForestClassifier

## 1. Load Data and Prepare Targets
We load our cleaned dataset from the `data` directory using `joblib`. The target variable `personal_loan` is mapped to binary integers, and non-predictive identifiers are dropped.

In [22]:
# Load the cleaned data using relative paths
df = joblib.load('../data/cleaned_data.pkl')

# Map the target variable
df['personal_loan'] = df['personal_loan'].map({'yes': 1, 'no': 0})

# Separate features (X) and target (y)
X = df.drop(['personal_loan', 'customer_id'], axis=1)
y = df['personal_loan']

## 2. Train/Test Split
To strictly prevent data leakage, all transformations must occur after splitting the data. We use an 80/20 split and apply stratification to maintain the class imbalance ratio in both sets.

In [23]:
# Perform the 80/20 train/test split with stratification
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Create explicit copies to prevent SettingWithCopyWarning
X_train = X_train.copy()
X_test = X_test.copy()
y_train = y_train.copy()
y_test = y_test.copy()

## 3. One-Hot Encoding
We systematically encode all categorical text variables into integer-based dummy variables. We are bypassing feature selection in this baseline notebook to test the algorithm on the complete, noisy dataset.

In [24]:
# Identify ALL categorical columns containing text
categorical_cols = ['education_level', 'credit_card_acct', 'online_acct']

# Loop through and apply pd.get_dummies() strictly preventing leakage
for col in categorical_cols:
    dummies_train = pd.get_dummies(X_train[col], prefix=col, dtype=int)
    X_train.drop([col], axis=1, inplace=True)
    X_train = pd.concat([X_train, dummies_train], axis=1)
    
    dummies_test = pd.get_dummies(X_test[col], prefix=col, dtype=int)
    X_test.drop([col], axis=1, inplace=True)
    X_test = pd.concat([X_test, dummies_test], axis=1)

# Align columns to guarantee train and test sets match perfectly
X_train, X_test = X_train.align(X_test, join='left', axis=1, fill_value=0)

print("All categorical variables successfully one-hot encoded into integers.")

All categorical variables successfully one-hot encoded into integers.


## 4. Train Baseline Model and Evaluate
We instantiate our Baseline Random Forest, train it on the all-features dataset, and evaluate its performance against the class imbalance.

In [25]:
# Instantiate and fit the baseline Random Forest model
rf_model_all = RandomForestClassifier(random_state=42)
rf_model_all.fit(X_train, y_train)

# Generate predictions on the test set
y_pred_all = rf_model_all.predict(X_test)

# Flatten the confusion matrix
tn, fp, fn, tp = confusion_matrix(y_test, y_pred_all).ravel()

# Calculate specific metrics
precision = precision_score(y_test, y_pred_all, zero_division=0)
recall = recall_score(y_test, y_pred_all)
f1 = f1_score(y_test, y_pred_all)

print("Baseline Random Forest Evaluation")
print("*" * 48)
print(f"True Negatives (TN):  {tn}")
print(f"False Positives (FP): {fp}")
print(f"False Negatives (FN): {fn}")
print(f"True Positives (TP):  {tp}")
print("*" * 48)
print(f"Precision: {precision:.4f}")
print(f"Recall:    {recall:.4f}")
print(f"F1-Score:  {f1:.4f}")

Baseline Random Forest Evaluation
************************************************
True Negatives (TN):  975
False Positives (FP): 45
False Negatives (FN): 57
True Positives (TP):  123
************************************************
Precision: 0.7321
Recall:    0.6833
F1-Score:  0.7069
